# Практическое занятие 1. REST API на примере OMDb (ветка env-feature)

В этой ветке API-ключ хранится в файле `.env` и читается через переменную
окружения. Файл `.env` добавлен в `.gitignore` и не попадает в репозиторий.

Перед запуском создайте в корне проекта файл `.env` со строкой:

```
OMDB_API_KEY=ваш_ключ
```

Код оформлен функциями и проверяет ответ API после code review.

## 1. Подключаем библиотеки

In [ ]:
import os
import json

import requests
from dotenv import load_dotenv

## 2. Функции: ключ, запрос и вывод

In [ ]:
OMDB_URL = "http://www.omdbapi.com/"


def get_api_key():
    """Читает ключ доступа из переменной окружения OMDB_API_KEY."""
    load_dotenv()
    api_key = os.getenv("OMDB_API_KEY")
    if not api_key:
        raise ValueError("Ключ OMDB_API_KEY не найден в .env")
    return api_key


def fetch_movie(title, api_key):
    """Запрашивает данные о фильме и возвращает разобранный JSON."""
    response = requests.get(
        OMDB_URL,
        params={
            "t": title,
            "apikey": api_key,
        },
        timeout=10,
    )
    response.raise_for_status()
    return response.json()


def print_movie_info(data):
    """Выводит поля фильма или понятное сообщение, что фильм не найден."""
    if data.get("Response") == "False":
        print("Фильм не найден:", data.get("Error"))
        return

    title = data.get("Title")
    year = data.get("Year")
    genre = data.get("Genre")
    director = data.get("Director")
    actors = data.get("Actors")
    rating = data.get("imdbRating")

    print(f"Название: {title}")
    print(f"Год: {year}")
    print(f"Жанр: {genre}")
    print(f"Режиссёр: {director}")
    print(f"Актёры: {actors}")
    print(f"Рейтинг IMDb: {rating}")
    print(f"Фильм '{title}' ({year}) — жанр: {genre}. Режиссёр: {director}.")

## 3. Запуск: ищем любимый фильм

Проверяем три случая: существующий фильм, несуществующее название и
неверный API-ключ.

In [ ]:
title = input("Введите фильм для поиска: ")

try:
    api_key = get_api_key()
    json_data = fetch_movie(title, api_key)
except requests.HTTPError as error:
    # Например, сервис вернул "Invalid API key!"
    print("Ошибка доступа к OMDb API:", error)
except requests.RequestException as error:
    print("Не удалось выполнить запрос:", error)
else:
    print(json.dumps(json_data, ensure_ascii=False, indent=2))
    print()
    print_movie_info(json_data)